In [3]:
import pandas as pd
import csv
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

def load_data():
    #['tree_data', 'rescale_factor', 'internal_nodes_order', 'ancestor', 'mutation_positions', 'number_of_mutantes', 
    # 'time_of_surgimento_mutacao', 'total_time_of_simulation', 'simulacoes_com_erro', 'index_order', 'tabela_parametros']
    data = np.load('C:\\Users\\JPC\\Documents\\MESTRADO\\Projeto\\gillespie-2\\mutation_deep_learning.npz', allow_pickle=True)
    #simulacoes_com_erro = [59295 59562 95849]
    trees_cblv = data['tree_data']
    trees_cblv = np.delete(trees_cblv, data['simulacoes_com_erro'], axis=0)

    #R_0_1, infectuos time and a taxa relativa 
    # Retornando a segunda coluna (coluna referente ao R_nought)
    # e em segunda transformando a lista em um np.array

    tabela_de_parametros = data['tabela_parametros']
    tabela_de_parametros = np.delete(tabela_de_parametros, data['simulacoes_com_erro'], axis=0)

    R_nought1 = np.array([sublist[0] for sublist in tabela_de_parametros])
    R_nought2 = np.array([sublist[1] for sublist in tabela_de_parametros])
    infectious_time = np.array([sublist[8] for sublist in tabela_de_parametros])
    proportion_tr11_tr22 = np.array([sublist[10] for sublist in tabela_de_parametros])
    
    rescale_factors = np.array(data['rescale_factor'])
    rescale_factors = np.delete(rescale_factors, data['simulacoes_com_erro'])   

    infectious_time_div_reascale = np.round(infectious_time / rescale_factors, 3)

    vectorized = np.column_stack((R_nought1, infectious_time_div_reascale, proportion_tr11_tr22))

    return trees_cblv, vectorized, rescale_factors



In [5]:
X, y, rescale_factors = load_data()

In [6]:
test_size = 10000
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    shuffle=False
)

In [5]:
scaler = StandardScaler()
y_train = scaler.fit_transform(y_train)
y_test = scaler.transform(y_test)

In [6]:
modelo_derivado = tf.keras.models.Sequential([
    layers.Input(X_train[0].shape),
    layers.Reshape((501, 2)),
    layers.Conv1D(filters = 50, kernel_size = 3, activation='relu'),
    layers.Conv1D(filters = 50, kernel_size = 10, activation='relu'),
    layers.MaxPooling1D(pool_size=10),
    layers.Conv1D(filters = 80, kernel_size = 10, activation='relu'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='elu'),
    layers.Dense(32, activation='elu'),
    layers.Dense(16, activation='elu'),
    layers.Dense(8, activation='elu'),
    #layers.Dense(2, activation='elu'),
    layers.Dense(3, activation='linear')
    ])



In [7]:
modelo_derivado.compile(optimizer='adam',loss='mae',metrics=['mae'])
modelo_derivado.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 501, 2)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 499, 50)        │           350 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 490, 50)        │        25,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 49, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 40, 80)         │        40,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 80)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         5,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 73,435 (286.86 KB)

 Trainable params: 73,435 (286.86 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = modelo_derivado.fit(X_train, y_train, epochs=25, batch_size=32, validation_split=0.1)

Epoch 1/25
2532/2532 ━━━━━━━━━━━━━━━━━━━━ 108s 41ms/step - loss: 0.5411 - mae: 0.5411 - val_loss: 0.4459 - val_mae: 0.4459
Epoch 2/25
2532/2532 ━━━━━━━━━━━━━━━━━━━━ 102s 40ms/step - loss: 0.4366 - mae: 0.4366 - val_loss: 0.4407 - val_mae: 0.4407
Epoch 3/25
2532/2532 ━━━━━━━━━━━━━━━━━━━━ 103s 41ms/step - loss: 0.4110 - mae: 0.4110 - val_loss: 0.3955 - val_mae: 0.3955
Epoch 4/25
2532/2532 ━━━━━━━━━━━━━━━━━━━━ 103s 41ms/step - loss: 0.3970 - mae: 0.3970 - val_loss: 0.4001 - val_mae: 0.4001
Epoch 5/25
2532/2532 ━━━━━━━━━━━━━━━━━━━━ 101s 40ms/step - loss: 0.3882 - mae: 0.3882 - val_loss: 0.3929 - val_mae: 0.3929
Epoch 6/25
2532/2532 ━━━━━━━━━━━━━━━━━━━━ 106s 42ms/step - loss: 0.3827 - mae: 0.3827 - val_loss: 0.3831 - val_mae: 0.3831
Epoch 7/25
2532/2532 ━━━━━━━━━━━━━━━━━━━━ 105s 42ms/step - loss: 0.3795 - mae: 0.3795 - val_loss: 0.3819 - val_mae: 0.3819
Epoch 8/25
2532/2532 ━━━━━━━━━━━━━━━━━━━━ 104s 41ms/step - loss: 0.3744 - mae: 0.3744 - val_loss: 0.3771 - val_mae: 0.3771
Epoch 9/25
2532/

In [9]:
modelo_derivado.save("model_4-05_normalizado.keras")

In [10]:
pred = modelo_derivado.predict(X_test)
pred = scaler.inverse_transform(pred)
pred[:, 1] *= rescale_factors[-test_size:]

y_test = scaler.inverse_transform(y_test)
y_test[:, 1] *= rescale_factors[-test_size:]

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step


In [11]:
erros = np.abs(pred - y_test)
mae_por_variavel = np.mean(erros, axis=0)

print("MAE R0:", mae_por_variavel[0])
print("MAE infectious_time:", mae_por_variavel[1])
print("MAE proportion:", mae_por_variavel[2])

MAE R0: 0.32637048814482483
MAE infectious_time: 0.38820484256119725
MAE proportion: 0.27033267569055597


In [12]:
erro_relativo = np.mean(np.abs((pred - y_test) / y_test), axis=0)

print("Erro relativo R0:", erro_relativo[0])
print("Erro relativo time:", erro_relativo[1])
print("Erro relativo proportion:", erro_relativo[2])

Erro relativo R0: 0.12262790548792886
Erro relativo time: 0.06944747983414659
Erro relativo proportion: 0.12414198269945925


In [13]:
print(np.mean(np.abs(pred - y_test), axis=0))

[0.32637049 0.38820484 0.27033268]


In [14]:
print(np.corrcoef(y_test[:,0], pred[:,0])[0,1])
print(np.corrcoef(y_test[:,1], pred[:,1])[0,1])
print(np.corrcoef(y_test[:,2], pred[:,2])[0,1])

0.9175833830925129
0.9783797971944131
0.6148760248364893
